# 02 · FFA-Net 通道 / 像素注意力

拆解通道注意力、像素注意力与多组特征融合；blocks/dim 可调小以便快速验证。

In [ ]:
import torch
import torch.nn as nn


class CALayer(nn.Module):
    def __init__(self, channel, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.ca = nn.Sequential(
            nn.Conv2d(channel, channel // reduction, 1), nn.ReLU(inplace=True),
            nn.Conv2d(channel // reduction, channel, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.ca(self.avg_pool(x))


class PALayer(nn.Module):
    def __init__(self, channel, reduction=8):
        super().__init__()
        self.pa = nn.Sequential(
            nn.Conv2d(channel, channel // reduction, 1), nn.ReLU(inplace=True),
            nn.Conv2d(channel // reduction, 1, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.pa(x)


class Block(nn.Module):
    def __init__(self, dim, kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv2d(dim, dim, kernel_size, padding=kernel_size // 2)
        self.act1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(dim, dim, kernel_size, padding=kernel_size // 2)
        self.ca, self.pa = CALayer(dim), PALayer(dim)

    def forward(self, x):
        res = self.act1(self.conv1(x)); res = res + x
        res = self.conv2(res); res = self.ca(res); res = self.pa(res)
        return res + x


class Group(nn.Module):
    def __init__(self, dim, blocks, kernel_size=3):
        super().__init__()
        self.gp = nn.Sequential(*[Block(dim, kernel_size) for _ in range(blocks)])
        self.gp.append(nn.Conv2d(dim, dim, kernel_size, padding=kernel_size // 2))

    def forward(self, x):
        return self.gp(x) + x


class FFANet(nn.Module):
    def __init__(self, gps=3, blocks=4, dim=64):
        super().__init__()
        self.gps, self.dim = gps, dim
        self.pre = nn.Conv2d(3, dim, 3, padding=1)
        self.groups = nn.ModuleList([Group(dim, blocks) for _ in range(gps)])
        self.fuse = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(dim * gps, dim // 16, 1), nn.ReLU(inplace=True),
            nn.Conv2d(dim // 16, dim * gps, 1), nn.Sigmoid())
        self.pa = PALayer(dim)
        self.post = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1), nn.Conv2d(dim, 3, 3, padding=1))

    def forward(self, x):
        feat = self.pre(x)
        feats = []
        for g in self.groups:
            feat = g(feat); feats.append(feat)
        w = self.fuse(torch.cat(feats, dim=1)).view(-1, self.gps, self.dim, 1, 1)
        out = sum(w[:, i] * feats[i] for i in range(self.gps))
        out = self.pa(out)
        return x + self.post(out)


In [ ]:
model = FFANet(gps=3, blocks=4, dim=32)
x = torch.randn(1, 3, 256, 256)
y = model(x)
print("输入:", x.shape, "输出:", y.shape)
print("参数量: %.2f M" % (sum(p.numel() for p in model.parameters()) / 1e6))
